In [ ]:
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import os
import torchvision.transforms.functional as TF
import random
import shutil
import cv2
import numpy as np
from PIL import Image
import glob
import timm

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from timm.models.vision_transformer import Attention
from transformers import AutoModel

#this is how you use your own data in google drive
from google.colab import drive


#define device - put this in the main func
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1 . Mount the Google Drive
drive.mount('/content/drive')



# 2. path for saving the weights
weights_path = "/content/drive/MyDrive/FinalProject_CSC2503/DINOv2_weights.pth"


# CREATING A DINOv2 PipeLine for fracture classification
# using the unsupervised embeddings to train a supervised classifier for the fractures


################ NEW VISUALIZATION FUNCTIONS ###################

## Directory folder to save the visualization plots
visualization_dir_plots = "/content/drive/MyDrive/FinalProject_CSC2503/DINOv2__pretrained_Visualizations_Plots"


def visualize_attention_patch(imgs, backbone, labels, predictions, class_names, device):
  # imgs [B, C, H, W] B =BATCH C=CHANNELS
  # backbone = DINOv2 pretrained model - set output_attentions=True
  # predictions predicted class indicies

  # backbone is model in eval mode - no updating of weights, turns off dropout, stops updating means/vars
  # layers must be stable when visualizing thts why we freeze it in eval mode
  backbone.eval()

  with torch.no_grad(): # SAVES MEMORY IN COLAB bc python not tracking weights
    # 1. sends imgs thru DINOv2, gets embeddings + attention maps
    #    outputs object contains: outputs.last_hidden_state, outputs.attentions
    outputs = backbone(imgs.to(device))
    # 2. token embeddings for patchs made by ViT - the +1 is the special class token, D=embed_dim(ie.768), N=num img patches
    hidden = outputs.last_hidden_state # [B, N+1, D] N=num img patchs, +1 for class token
    # 3. list of attn matrices, one for each transformer layer (each layer has mulitple attn heads)
    attentions = outputs.attentions # [12 heads, 197 tokens, 197 tokens] - heatmaps tht each transformer layer focusses on

  B = imgs.shape[0]

  # remove the class token and get patch embeddings
  patch_tokens = hidden[:, 1:, :] # start at 1 bc classtoken is at 0 [B, N+1, D]
  patch_tokens = F.normalize(patch_tokens, dim=-1) # normalize before cosine similarity
  # get patch dim ie 14x14
  num_patches = int(patch_tokens.shape[1] **0.5) #**0.5 is ^(1/2) so ITS A SQR ROOT to get patch grid size-> sqrt(196)=14

  # compute similarity to each path
  cls_token = hidden[:, 0, :]
  cls_token = F.normalize(cls_token, dim=-1) # normalize to unit length (ie. 1) before cos similarity

  #einsum does dot prod btw CLS vector and every patch vector
  # dot prod is NOT cos similiarity BUT it equals cos similiars IFF both vectors are normalized to UNIT LENGTH (ie. 1)
  #"bp" labels cls_token axes as (batch, embed-dim), "bpd" labels patch_tokens axes as (batch, patch index, embed_dim), -> bp produces an output w axes (batch, patch_index)
  # for each batch b and patch p, einsum multiplies cls_token[b, d]*patch_tokens[b, p, d] and sums over d which equals dot prod of 2 vectors; resulting shape is [B, P]

  #similarity=torch.einsum("bd, bpd -> bp", cls_token, patch_tokens)

  #COSINE SIMILARITY HERE FOR THE PATCHES
  similarity = F.cosine_similarity(
    patch_tokens,                  # (B, P, D)
    cls_token.unsqueeze(1),        # (B, 1, D) → broadcasts to (B, P, D)
    dim=-1                         # similarity along embedding dimension
  )

  similarity = similarity.reshape(B, num_patches, num_patches) # shape is (Barch, num_patches, num_patches) -resize to height of num_patches and width of num_patches
  similarity = similarity.cpu().numpy()


  # VISUALIZE EVERYTHING NOW

  for i in range(B): # B=batch size
    # convert tensor  to numpy image
    img_np = imgs[i].cpu().permute(1, 2, 0).numpy() #imgs[i] is ith img shape [3, H, W]; .permute moves channels at index 0 to the last spot; cpu().numpy() moves from gpu to cpu
    # normalize img values bwt [0, 1 for imshow() display
    img_np = (img_np - img_np.min()) /(img_np.max() - img_np.min())


    # similarity: numpy array of shape (Batch, num_patches, num_patches)
    # attentions: list of length num_layers, each element shape (Batch, num_heads, tokens, tokens)
    # imgs: tensor shape (Batches, channels, H, W)


    # FOR-LOOP PER IMAGE INDEX i

    ## SIMILARITY MAP ##
    similarity_map = similarity[i] # get patch similarity for ith image; similarity.shape = (num_patches, num_patches)
    similarity_map = (similarity_map - similarity_map.min())/(similarity_map.max() - similarity_map.min()) # normalize patch similarity map to [0, 1]


    # upsample patch similarity to image size so we can overlay it on top of acc img
    # imgs.shape = [Batch, channel, H, W] thus imgs[2]=224 for H
    upsample_factor = imgs.shape[2] // num_patches # imgs.shape[2]=224, num_patches =14 (each patch corresponds to 16x16 pixels)
    # np.kron repeats each value into a block, it turns 14x14=196 patches of 16x16 pixels into a 224x224 heatmap
    # two arrays A=similarity_map=(mxn) B=np.ones=(pxq), np.kron does (m*p)x(n*q) multiplies each element in A by entire matrix B
    #imgs.shape[3] = W and imgs.shape[2] = H
    similarity_up =cv2.resize(similarity_map, (imgs.shape[3], imgs.shape[2]), interpolation=cv2.INTER_CUBIC)


    ## ATTENTION MAPS ##

    # get attention from shallow layer 0 attention.shape is [num_layers][batch][num_heads][tokens][tokens]
    layer0 = attentions[0][i] # layer0 shape is [num_heads, tokens, tokens]
    attn = layer0.mean(dim=0) # .mean to get one attn matrix shape [tokens, tokens] (an average over all attention heads, instead of having N heads)
    cls_attn = attn[0, 1:] # index 0=class token, 1..P=patch tokens; cls attends to patch 1, 2 .. P

    cls_attn = cls_attn.reshape(num_patches, num_patches).cpu().numpy() # reshape to pathc grid, it turns 196 to (14x14)
    cls_attn = (cls_attn - cls_attn.min())/(cls_attn.max() - cls_attn.min() +1e-8) #normalize attn to [0,1]
    cls_attn_up = cv2.resize(cls_attn, (imgs.shape[3], imgs.shape[2]), interpolation=cv2.INTER_CUBIC) # upsample to img dimensions


    # PLOT ATTENTION MAPS, PATCH SIMILARITY
    plt.figure(figsize=(15,5))

    #### NEED TO CONFIRM IF THESE R BEING PASSED CORRECTLY
    true_label = class_names[labels[i]]
    predicted_label = class_names[predictions[i]]


    ## EACH PLOT HAS 3 PANELS SIDE BY SIDE
    # 1. Original image
    plt.subplot(1,3,1)
    plt.imshow(img_np)
    plt.title(f"Original\nGT {true_label}\nPredicted: {predicted_label}")
    plt.axis('off')

    # 2. Attention map (CLS → patches)
    plt.subplot(1,3,2)
    plt.imshow(img_np) # orig image
    plt.imshow(cls_attn_up, cmap='inferno', alpha=0.5) # heatmap on top of orig image
    plt.title("DINOv2 Attention Map\n(How Much the CLS Token Attends to Each Patch)")
    plt.axis('off')

    # 3. Patch similarity map (CLS similarity)- what patches look similar to the global concept
    plt.subplot(1,3,3)
    plt.imshow(img_np) #orig image
    plt.imshow(similarity_up, cmap='viridis', alpha=0.5) # patch similarity on top of orig image
    plt.title("Patch Similarity Map\n(CLS similarity)")
    plt.axis('off')

    # SAVE THE PLOTS HERE
    save_path = os.path.join(visualization_dir_plots, f"image_{i}.png")
    plt.savefig(save_path, bbox_inches='tight')

    plt.show()





### PATHWAYS TO DATA ###

# contains original images (no preprocessing)
original_data_dir = "/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification"

# preprocessed images are saved here (resizing and padding)
processed_data_dir = "/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification Resized"

# the final image size that we want
target_img_size = 224



# ONLY RUN THIS FUNCTION ONCE!!!!!
# function calls the preprocess_img_resize_pad function for preprocessing and saves new imgs in a diff location
def preprocess_dataset():

  print("PREPROCESSING STARTED")
  print("")

  # looks for fracture type directories
  for fracture_type in os.listdir(original_data_dir):
    # gets the path for one fracture type
    fracture_path = os.path.join(original_data_dir , fracture_type)
    if not os.path.isdir(fracture_path):
      continue

    # checking that the train and test folders exist - nothing else has happened yet
    for split in ["Train", "Test"]:
      # path to the train or test folder
      split_path = os.path.join(fracture_path, split)
      if not os.path.exists(split_path):
        continue

      # saving in the new destination here
      save_dir = os.path.join(processed_data_dir, fracture_type, split)
      os.makedirs(save_dir, exist_ok = True)

      # looping thru each img file in either the train and test folder per fracture type
      for filename in os.listdir(split_path):

        # load the image
        path = os.path.join(split_path, filename)
        img = Image.open(path) #.convert("RGB")

        # apply the resizing and padding
        processed = preprocess_img_resize_pad(img, target_img_size)

        # save this processed image in the new directory
        save_path = os.path.join(save_dir, filename)
        processed.save(save_path)

        print("PREPROCESSING COMPLETE AND IMGS SAVED IN NEW LOCATION")





# ONLY RUN THIS FUNCTION ONCE!!!!!
# take 20% of the training data and put it in the val folder
def make_val_split(val_ratio=0.2):
  for fracture_type in os.listdir(processed_data_dir):
    # creating paths to the folders
    class_dir = os.path.join(processed_data_dir, fracture_type)
    train_dir = os.path.join(class_dir, "Train")
    val_dir = os.path.join(class_dir, "Val")

    os.makedirs(val_dir, exist_ok=True)

    # CHECK IF WEVE ALREADY MADE THE VAL FOLDER
    if len(os.listdir(val_dir)) > 0:
      print(f"Skipping {fracture_type}: val already created.")
      continue


    images = os.listdir(train_dir)
    random.shuffle(images) # random selection of the images


    val_count = int(len(images)*val_ratio) #val_ratio=0.2; this is where the 20% split happens
    val_images = images[:val_count]

    # moving the images to the val folder
    for img in val_images:
      src = os.path.join(train_dir, img)
      dst = os.path.join(val_dir, img)
      shutil.move(src, dst)



    print(f"{fracture_type}: moved {val_count} images to val/")


############## HELPER FUNCTIONS FOR THE DATALOADER, REFORMATTING THE IMAGE PATHS SO IT CAN BE USED PROPERLY - SAME CODE FROM GRAD-CAM, SCORE-CAM #####

### obtain all images within the 'train' or 'val' subfolder of the  fracture type folder ###
def build_image_list(root, fracture_classes, split): #root= dataset path, split = 'train' or 'val'
  items = [] # holds (image_path, label_index) tupels
  for idx, cls in enumerate(fracture_classes): #idx is integer label assigned to the class in fracture_classes list; cls is class name string ("Avulsion fracture")
    folder = os.path.join(root, cls, split) # creates custom path bc root = "/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification", cls = "Avulsion fracture", split = 'train' or 'val'
    if not os.path.isdir(folder):
      raise FileNotFoundError(f"Expected {folder} to exist")
    # if we do have a folder path, continue on looping thru images
    for ext in('*.png', '*.jpg', '*.jpeg'):
      for p in glob.glob(os.path.join(folder, ext)): # finds all matching file paths (p) in this folder
        items.append((p, idx)) # append file path p and numerical class integer label idx to the list items

  # return looks like this: ("/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification/Avulsion fracture/train/image_123.jpg", 0) -> 0 is the numerical integer idx for 'Avulsion fracture' folder
  return items # has the path and numerical label for the requested split - 'train' or 'val'

###   converts to RGB, APPLYS TRANSFORMS AND RETURNS A TUPLE ###
class ImageListDataset(Dataset):
  def __init__(self, items, transform=None):
    self.items = items # items is list return by build_image_list
    self.transform = transform
  def __len__(self):
    return len(self.items) # return # of images we have
  def __getitem__(self, idx):
    path, label = self.items[idx] # retirieve the tuple (path, label) from items list, label is numerical integer label
                                  # path is specific: /content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification/Avulsion fracture/train/img_045.jpg
    img = Image.open(path)#.convert('RGB') # convert to RGB bc most pretrained CNNs expect 3 channel input otherwise we'd get mismatch error
                                          # if we were training from scratch then we dont need to convert to RGB
    # apply transforms here
    if self.transform:
      img = self.transform(img)

    # img is torch.Tensor (3x224x224), label is int (0-9), path is str to the img loc ie./.../Avulsion fracture/train/img_001.jpg for saving gradCAM overlay
    return img, label, path # path so we can save GRAD-CAM overlays next to source images

#######################################################################################################################


# RUN PREPROCESS DATASET ONCE BEFORE TRAINING
# COMMENT THIS LINE OUT ONCE YOUVE DONE IT ONCE
#preprocess_dataset()

# run only once
#make_val_split(val_ratio=0.2)


# can define dataloaders up here
transform = T.Compose([T.Resize(size=(224, 224)),
                       T.Grayscale(num_output_channels=3),
                       T.ToTensor(), T.Normalize(mean=[0.5]*3, std = [0.5]*3)])


# 10 classes of fracture here
fracture_classes = ['Avulsion fracture',
               'Comminuted fracture',
               'Fracture Dislocation',
               'Greenstick fracture',
               'Hairline Fracture',
               'Impacted fracture',
               'Longitudinal fracture',
               'Oblique fracture',
               'Pathological fracture',
               'Spiral Fracture']

# Load the entire fracture dataset (images + labels)
#fracture_dataset = torchvision.datasets.ImageFolder(root=processed_data_dir, transform=transform)
#print("Class mapping:", fracture_dataset.class_to_idx)


train_items = build_image_list(processed_data_dir, fracture_classes, split='Train')
val_items = build_image_list(processed_data_dir, fracture_classes, split='Val')
test_items = build_image_list(processed_data_dir, fracture_classes, split='Test')

train_dataset = ImageListDataset(train_items, transform=transform)
val_dataset = ImageListDataset(val_items, transform=transform)
test_dataset = ImageListDataset(test_items, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers = 2, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers = 2, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers = 2, pin_memory = True)


# print # images for train and val sets
print("Train samples:", len(train_dataset), "Val samples:", len(val_dataset), "Test samples:", len(test_dataset))



# TRAINING FUNCTION
def train_epoch(model, linear_head, loader, optimizer, device):
  model.eval() # PUT IN EVAL MODE TO FREEZE THE UNSUPERVISED BACKBONE DURING LINEAR PROBING
  total_loss = 0
  correct = 0

  for imgs, labels, _ in loader:
    imgs, labels = imgs.to(device), labels.to(device)


    with torch.no_grad(): # THIS FREEZES BACKBONE TOO BC ITS PART OF EVAL
      outputs = model(imgs) # shape = [B, embedding_dim] B = batch
      embeddings = outputs.last_hidden_state
      class_embed = embeddings[:, 0, :] # CLASS TOKEN




    # randomize the incoming batch, save it to imgs2 and then pass into model
    # randomizing will make the imgs and img2 batch order diiferent so respective images can be paired together for vitmix
    #imgs2 = imgs[torch.randperm(imgs.size(0))]


    logits = linear_head(class_embed)
    loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()*imgs.size(0)
    correct += (logits.argmax(dim=1) == labels).sum().item()

  avg_loss = total_loss/len(loader.dataset)
  accuracy = correct/len(loader.dataset)

  return avg_loss, accuracy




# EVALUATION FUNCTION
def evaluate(model, linear_head, loader, device):

  model.eval()
  linear_head.eval() # PUTTING THE LINEAR_HEAD INTO EVAL MODE TOO

  total_loss = 0
  correct = 0

  with torch.no_grad():
    for imgs, labels, _ in loader:
      imgs, labels = imgs.to(device), labels.to(device)

      outputs = model(imgs) # shape = [B, embedding_dim] B = batch
      embeddings = outputs.last_hidden_state
      class_embed = embeddings[:, 0, :] # CLASS TOKEN

      # raw, unormalized outputs of neural net (BEFORE softmax or sigmoid is applied)
      logits = linear_head(class_embed)

      # cross entropy expects logits bc it has a softmax inside it
      loss = F.cross_entropy(logits, labels)

      total_loss += loss.item()*imgs.size(0)
      # the logits.argmax(dim=1) == labels returns smth like tensor([True, True, True])
      correct += (logits.argmax(dim=1) == labels).sum().item()

  avg_loss = total_loss/len(loader.dataset)
  accuracy = correct/len(loader.dataset)

  return avg_loss, accuracy



# TEST FUNCTION
def test(model, linear_head, test_loader, device, fracture_classes):
  model.eval()
  linear_head.eval() # PUTTING THE LINEAR_HEAD INTO EVAL MODE TOO

  total_loss = 0.0
  correct = 0

  with torch.no_grad():
    for imgs, labels, _ in test_loader:
      imgs, labels = imgs.to(device), labels.to(device)

      outputs = model(imgs) # shape = [B, embedding_dim] B = batch
      embeddings = outputs.last_hidden_state
      class_embed = embeddings[:, 0, :] # CLASS TOKEN

      logits = linear_head(class_embed)
      loss = F.cross_entropy(logits, labels)

      # predictions
      preds = logits.argmax(dim=1)

      total_loss += loss.item()*imgs.size(0)
      correct += (preds == labels).sum().item()

      # visualize attention maps and patch similarity here
      visualize_attention_patch(imgs, model, labels, preds, fracture_classes, device)


  avg_loss = total_loss/len(test_loader.dataset)
  accuracy = correct/len(test_loader.dataset)

  print(f"Test Loss: {avg_loss:.4f} | Test Accuracy: {accuracy:.4f}")





# RUN THE TRAINING LOOP
dino = AutoModel.from_pretrained("facebook/dinov2-base", output_attentions=True) # lighter model for colab
model = dino.to(device)  # or ViTMiX version
# Define the linear classification head - 10 classification possibilities
linear_head = nn.Linear(768, len(fracture_classes)).to(device)
optimizer = torch.optim.Adam(linear_head.parameters(), lr=1e-4)
best_val_acc = 0



####### CHECK IF WEIGHTS WERE SAVED
start_epoch = 0
best_val_acc = 0.0
epochs = 20  # change this if ur restarting


# Optionally load weights if continuing training or evaluating
if os.path.exists(weights_path):

  #state = torch.load(weights_path, map_location=device)

  checkpoint = torch.load(weights_path, map_location=device)
  #model.load_state_dict(checkpoint['model_state_dict'])
  linear_head.load_state_dict(checkpoint['linear_state_dict'])
  optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
  start_epoch = checkpoint['epoch'] + 1 # resume from next epoch
  best_val_acc = checkpoint['val_acc']
  print(f"✅ Loaded checkpoint from epoch {start_epoch}")
  print("Models loaded successfully!")

else:
  print("NO previously saved models found!")




# BEGIN LOOPING THRU EPOCHS
for epoch in range(start_epoch, epochs):
  train_loss, train_acc = train_epoch(model, linear_head, train_loader, optimizer, device)
  val_loss, val_acc = evaluate(model, linear_head, val_loader, device)

  print(f"[Epoch {epoch+1}] "
          f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
          f"Val Loss {val_loss:.4f} Acc {val_acc:.4f}")

  # Save best model
  if val_acc > best_val_acc:
    best_val_acc = val_acc


    # saving epochs, optimizer, state
    torch.save({
      'epoch': epoch,
      #'model_state_dict': model.state_dict(),
      'linear_state_dict': linear_head.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'val_acc': val_acc
      }, weights_path)

    print(f"✅ Saved best model (epoch {epoch+1}) with val_acc={val_acc:.4f} to {weights_path}")




# RUN TEST FUNCTION ONCE TRAINING AND EVALUATION ARE DONE
# VISUALIZATION FUNCTION IS WITHIN THIS ALREADY
test(model, linear_head, test_loader, device, fracture_classes)




